# DragonHPC ProcessGroup Tutorial for MPI Orchestration

**Estimated time:** ~40-55 minutes  
**Format:** 5 exercises, each with demo code, a short coding problem, and hidden solution.

One of the most important aspects of an HPC or AI workflow is asynchronous orchestration of processes. This could be for ensembles of modeling and simulation applications or large scale data processing, for example. Processes could be Python running a function, some serial executable (compiled or even Python itself), and parallel applications (e.g., using MPI or NCCL). Dragon supports orchestrating any of these types of processes through the [`ProcessGroup`](https://dragonhpc.github.io/dragon/doc/_build/html/ref/native/dragon.native.process_group.ProcessGroup.html#dragon.native.process_group.ProcessGroup) API.

To aid your intuition, recall that Dragon is deployed on some set of resources (laptop or thousands of nodes on a supercomputer). Dragon puts a minimal set of runtime components on those resources and otherwise leaves them idle for the user's workload. `ProcessGroup` can be used manage the life-cycle of many different types of applications simulataneously on the resources available to the runtime without relying on any other launch mechanism such as Slurm's `srun`. By using `ProcessGroup` for orchestration, all of the features of Dragon are avilable to all of the managed processes and there is no dependence on system specific tools like workload managers. This results in portable and self-contained workflows.

## Session goals

This tutorial introduces DragonHPC `ProcessGroup` orchestration for Python functions, serial executables, and MPI applications.

You will practice how to:

- create `ProcessGroup` for functions, executables, and parallel (e.g., MPI) applications with [`ProcessTemplate`](https://dragonhpc.github.io/dragon/doc/_build/html/ref/native/dragon.native.process.html#dragon.native.process.ProcessTemplate)
- launch MPI applications with [PMI backends](https://dragonhpc.github.io/dragon/doc/_build/html/ref/inf/dragon.infrastructure.facts.html#dragon.infrastructure.facts.PMIBackend)
- run several MPI applications at the same time
- manage stdout and output files
- control placement with [`Policy`](https://dragonhpc.github.io/dragon/doc/_build/html/ref/dragon.infrastructure.policy.Policy.html#dragon.infrastructure.policy.Policy)
- run MPI applications through higher-level [`Batch`](https://dragonhpc.github.io/dragon/doc/_build/html/ref/workflows/dragon.workflows.batch.Batch.html#dragon.workflows.batch.Batch) API

The exercises are designed for all levels and focus on practical workflow
patterns commonly used in HPC automation.

---

## Setup - run this first

Start Jupyter from a Dragon-enabled environment (for example with
`dragon-jupyter`) and run the next code cell.

In [1]:
import os
import sys

import dragon
import multiprocessing as mp

from dragon.infrastructure.facts import PMIBackend
from dragon.infrastructure.policy import Policy
from dragon.native.process import Process, ProcessTemplate, MSG_PIPE, MSG_DEVNULL
from dragon.native.process_group import ProcessGroup
from dragon.native.machine import System, current
from dragon.workflows.batch import Batch

mp.set_start_method("dragon")
print("Dragon ProcessGroup tutorial setup complete")

Dragon ProcessGroup tutorial setup complete


---

## Exercise 1 - Orchestrate Python functions with ProcessGroup

**Background:**

`ProcessGroup` can launch many copies of a Python function template and
coordinate lifecycle through `init`, `start`, `join`, and `close`.

Demo pattern:

```python
def hello():
    print("hello from", os.getpid())

pg = ProcessGroup()
pg.add_process(nproc=3, template=ProcessTemplate(target=hello))
pg.init()
pg.start()
pg.join()
pg.close()
```

**Your task:**

1. Write `ranked_hello(rank)` that prints rank (0-N index of the process), pid, and hostname
2. Build a ProcessGroup with 4 processes using one ProcessTemplate
3. Start, join, and close the group

In [9]:
# -- Exercise 1 -- your code here -----------------------------------------------

# Hint: the current() method imported from dragon.native.machine above has a property for hostname
def ranked_hello(rank):
    print(f"rank={rank} pid={os.getpid()} host={current().hostname}")

pg = ProcessGroup()
for rank in range(4):
    pg.add_process(nproc=1, template=ProcessTemplate(target=ranked_hello, args=(rank,)))

pg.init()
pg.start()
pg.join()
pg.close()

rank=3 pid=4288 host=localhost
rank=0 pid=4285 host=localhost
rank=1 pid=4286 host=localhost
rank=2 pid=4287 host=localhost


<details>
<summary><b>▶ Show Solution</b></summary>

```python
def ranked_hello(rank):
    print(f"rank={rank} pid={os.getpid()} host={current().hostname}")

pg = ProcessGroup()
for rank in range(4):
    pg.add_process(nproc=1, template=ProcessTemplate(target=ranked_hello, args=(rank,)))

pg.init()
pg.start()
pg.join()
pg.close()
```

</details>

---

## Exercise 2 - Run a serial executable and manage I/O

**Background:**

`ProcessTemplate` supports output routing. You can capture one process output
to `MSG_PIPE` and write others to files or `DEVNULL`.

Demo pattern:

```python
pg = ProcessGroup()
pg.add_process(nproc=1, template=ProcessTemplate(target="/bin/hostname", stdout=MSG_PIPE))
pg.init(); pg.start(); pg.join()
for puid, exit_code in pg.inactive_puids:
    proc = Process(None, ident=puid)
    if proc.stdout_conn:
        print(proc.stdout_conn.recv())
pg.close()
```

**Your task:**

1. Create a ProcessGroup with two templates running `/bin/echo`
2. First template: one process, output to `MSG_PIPE`
3. Second template: three processes, output to files `echo_0.txt`, `echo_1.txt`, `echo_2.txt`
4. Print the piped output and then print one file's contents
5. Join and close the group

In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------

# Hint: target="/bin/echo", args=("hello from template",)

<details>
<summary><b>▶ Show Solution</b></summary>

```python
output_files = [f"echo_{i}.txt" for i in range(3)]
pg = ProcessGroup()

pg.add_process(
    nproc=1,
    template=ProcessTemplate(target="/bin/echo", args=("hello from pipe",), stdout=MSG_PIPE),
)

for i in range(3):
    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target="/bin/echo", args=(f"hello file {i}",), stdout=output_files[i]),
    )

pg.init()
pg.start()
pg.join()

ofidx = 0
for puid, ecode in pg.inactive_puids:
    proc = Process(None, ident=puid)
    if proc.stdout_conn:
        print("PIPE:", proc.stdout_conn.recv().strip())
    else:
        fn = output_files[ofidx]
        ofidx += 1
        with open(fn, "r") as f:
            print(f"{fn}:", f.read().strip())
        os.unlink(fn)

pg.close()
```

</details>

---

## Exercise 3 - Launch an MPI application with ProcessGroup

**Background:**

In this exercise we will use the `mpi4py_example.py` script in the current directory as the MPI application. Your environment has `Open MPI` and `mpi4py` installed, which are required to run this application. It is a simple one dimensional Van Leer advection solver that uses MPI for exchange ghost zones between ranks, mimicking the behavior of common computational fluid dynamic applications.

To orchestrate an MPI executable, create a ProcessGroup with a [`PMI backend`](https://dragonhpc.github.io/dragon/doc/_build/html/ref/inf/dragon.infrastructure.facts.html#dragon.infrastructure.facts.PMIBackend)
and add process templates for rank groups. A common pattern is:

- rank 0 uses `MSG_PIPE` so we can parse output
- remaining ranks use `MSG_DEVNULL`

Demo pattern:

```python
exe = sys.executable  # alternatively could be compiled MPI application
pg = ProcessGroup(pmi=PMIBackend.PMIX)
pg.add_process(nproc=1, template=ProcessTemplate(target=exe, args=("mpi4pi_example.py",), stdout=MSG_PIPE))
pg.add_process(nproc=3, template=ProcessTemplate(target=exe, args=("mpi4pi_example.py",), stdout=MSG_DEVNULL))
pg.init(); pg.start(); pg.join(); pg.close()
```

**Your task:**

1. Launch `mpi4pi_example.py` with 4 total ranks using the two-template pattern. Note that `sys.executable` is the executable for `ProcessGroup` to run.
2. Capture and print output from the piped rank
3. Join and close the ProcessGroup

In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------

<details>
<summary><b>▶ Show Solution</b></summary>

```python
exe = sys.executable

pg = ProcessGroup(pmi=PMIBackend.PMIX)
pg.add_process(nproc=1, template=ProcessTemplate(target=exe, args=("mpi4py_example.py",), stdout=MSG_PIPE))
pg.add_process(nproc=3, template=ProcessTemplate(target=exe, args=("mpi4py_example.py",), stdout=MSG_DEVNULL))

pg.init()
pg.start()

for puid in pg.puids:
    proc = Process(None, ident=puid)
    if proc.stdout_conn:
        try:
            while True:
                line = proc.stdout_conn.recv()
                print(line.strip())
        except EOFError:
            pass

pg.join()
pg.close()
```

</details>

---

## Exercise 4 - Orchestrate several MPI jobs at the same time

**Background:**

HPC workflows often run multiple MPI applications concurrently. One practical
pattern is a producer function that creates and runs one `ProcessGroup`, then a
parent process launches multiple producers in parallel.

Demo pattern:

```python
def run_one_job(job_id, result_q):
    # create ProcessGroup for this job and redirect rank 0's output to a MSG_PIPE
    # start/join/close
    result_q.put((job_id, "done"))

result_q = mp.Queue()
p0 = Process(target=run_one_job, args=(0, result_q))
p1 = Process(target=run_one_job, args=(1, result_q))
p0.start(); p1.start()
print(result_q.get()); print(result_q.get())
p0.join(); p1.join()
```

**Your task:**

1. Write `run_mpi_job(job_id, num_ranks, result_q)` using `ProcessGroup` + `mpi4py_example.py`
2. In each job, pipe rank 0 stdout and send one parsed line back via `result_q`
3. Launch two jobs concurrently with `num_ranks=4` and `num_ranks=6`
4. Print both results
5. Ensure both producer processes are joined

In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------

def run_mpi_job(job_id, num_ranks, result_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_mpi_job(job_id, num_ranks, result_q):
    exe = sys.executable
    pg = ProcessGroup(pmi=PMIBackend.PMIX)

    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target=exe, args=("mpi4py_example.py",), stdout=MSG_PIPE),
    )
    pg.add_process(
        nproc=max(0, num_ranks - 1),
        template=ProcessTemplate(target=exe, args=("mpi4py_example.py",), stdout=MSG_DEVNULL),
    )

    pg.init()
    pg.start()

    first_line = None
    for puid in pg.puids:
        proc = Process(None, ident=puid)
        if proc.stdout_conn:
            try:
                first_line = proc.stdout_conn.recv().strip()
            except EOFError:
                first_line = "<no output>"
            break

    pg.join()
    pg.close()
    result_q.put((job_id, num_ranks, first_line))

result_q = mp.Queue()
p0 = Process(target=run_mpi_job, args=(0, 2, result_q))
p1 = Process(target=run_mpi_job, args=(1, 2, result_q))

p0.start()
p1.start()

print(result_q.get())
print(result_q.get())

p0.join()
p1.join()
```

</details>

---

## Exercise 5 - Placement control with Policy

**Background:**

Default placement spreads processes according to Dragon policy defaults
(default round-robin across available nodes). If you need explicit placement,
attach a `Policy` to `ProcessGroup` or `ProcessTemplate`.

Demo pattern:

```python
host = current().hostname
local_policy = Policy(placement=Policy.Placement.HOST_NAME, host_name=host)
template = ProcessTemplate(target="/bin/hostname", policy=local_policy, stdout=MSG_PIPE)
```

**Your task:**

1. Write `placement_probe(out_q)` that reports `(pid, hostname)`
2. Build one `Policy` pinned to current hostname
3. Launch 4 processes in a ProcessGroup using that policy
4. Collect and print all reports
5. Print how many unique hostnames were used (noting on a laptop their will only be one)

In [ ]:
# -- Exercise 5 -- your code here -----------------------------------------------

def placement_probe(out_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def placement_probe(out_q):
    out_q.put((os.getpid(), current().hostname))

q = mp.Queue()
host = current().hostname
local_policy = Policy(placement=Policy.Placement.HOST_NAME, host_name=host)

pg = ProcessGroup(restart=False)
template = ProcessTemplate(target=placement_probe, args=(q,), policy=local_policy)
pg.add_process(nproc=4, template=template)

pg.init()
pg.start()

reports = [q.get() for _ in range(4)]
for item in reports:
    print(item)

pg.join()
pg.close()

hosts = sorted({host for _, host in reports})
print("unique hosts:", hosts)
print("count:", len(hosts))
```

</details>

---

## BONUS Exercise 6 - Simplified Scheduling with Batch

**Background:**

When you find need of running far more processes than Dragon has resources for, you likely need a scheduler. `ProcessGroup` by itself simply runs what you ask when you say, but managing a finite set of resources and scheduling processes onto them is the job of another Dragon feature. Dragon's `Batch` API handles scheduling of functions, executables, and parallel applications on the resources available to a given instance of `Batch`. It has a rich set of features for data dependencies, error handling, and much more. Here we will use it to run an ensemble of `mpi4py_example.py` without oversubscribing the avilable CPU cores.

Demo pattern:

```python
tmpl = ProcessTemplate(target=sys.executable, args=("mpi4py_example.py",))
batch = Batch()
mpijobs = []
for _ in range(4):
    uid, job = batch.options(pmi=PMIBackend.PMIX).job(process_templates=[(nranks, tmpl)])
    mpijobs.append(job)

for job in mpijobs:
    try:
        exit_codes = job.get()
        print(f"Got exit codes {exit_codes}", flush=True)
    except Exception as e:
        print(f"Job got exception! {e}", flush=True)
batch.join()
batch.destroy()
```

**Your task:**

1. Modify the sample above to submit MPI jobs to `Batch` with different numbers of ranks, ie `[3, 2, 2, 12, 1]`
2. Use `job.cancel()` to cancel the job with 12 ranks, where `job` is a item in the `mpijobs` list

In [ ]:
# -- Exercise 6 -- your code here -----------------------------------------------
tmpl = ProcessTemplate(target=sys.executable, args=("mpi4py_example.py",))
batch = Batch()
sizes = [3, 2, 2, 12, 1]
mpijobs = []
for size  in sizes:
    job = batch.options(pmi=PMIBackend.PMIX).job(process_templates=[(size, tmpl)])
    mpijobs.append(job)

mpijobs[4].cancel()

for job in mpijobs:
    try:
        exit_codes = job.get()
        print(f"Got exit codes {exit_codes}", flush=True)
    except Exception as e:
        print(f"Job got exception! {e}", flush=True)
batch.join()
batch.destroy()

Rank 0 of 3 says: Hello from MPI!
Using CFL=0.8 and provided DDict=None
Step =      1 : SimTime = 1.0417e-03 : StepElapsed = 8.61e-03 s : CellsPS = 8.92e+04
Step =      2 : SimTime = 2.0833e-03 : StepElapsed = 9.75e-03 s : CellsPS = 7.88e+04
Rank 2 of 3 says: Hello from MPI!
Using CFL=0.8 and provided DDict=None
Rank 1 of 3 says: Hello from MPI!
Using CFL=0.8 and provided DDict=None
Step =      3 : SimTime = 3.1250e-03 : StepElapsed = 2.03e-02 s : CellsPS = 3.79e+04
Step =      4 : SimTime = 4.1667e-03 : StepElapsed = 9.49e-03 s : CellsPS = 8.09e+04
Step =      5 : SimTime = 5.2083e-03 : StepElapsed = 1.62e-02 s : CellsPS = 4.73e+04
Step =      6 : SimTime = 6.2500e-03 : StepElapsed = 1.64e-02 s : CellsPS = 4.67e+04
Step =      7 : SimTime = 7.2917e-03 : StepElapsed = 9.45e-04 s : CellsPS = 8.13e+05
Step =      8 : SimTime = 8.3333e-03 : StepElapsed = 1.38e-03 s : CellsPS = 5.57e+05
Step =      9 : SimTime = 9.3750e-03 : StepElapsed = 5.80e-02 s : CellsPS = 1.32e+04
Step =     10 : Sim

<details>
<summary><b>▶ Show Solution</b></summary>

```python
tmpl = ProcessTemplate(target=sys.executable, args=("mpi4py_example.py",))
batch = Batch()
sizes = [3, 2, 2, 12, 1]
mpijobs = []
for size  in sizes:
    job = batch.options(pmi=PMIBackend.PMIX).job(process_templates=[(size, tmpl)])
    mpijobs.append(job)

mpijobs[4].cancel()

for job in mpijobs:
    try:
        exit_codes = job.get()
        print(f"Got exit codes {exit_codes}", flush=True)
    except Exception as e:
        print(f"Job got exception! {e}", flush=True)
batch.join()
batch.destroy()
```

---

## Summary

You used ProcessGroup as an orchestration primitive for Python and MPI jobs,
including concurrent launches, output handling, and placement control.

| Concept | API |
|---|---|
| Define work | `ProcessTemplate(...)` |
| Group orchestration | `ProcessGroup.add_process(...), init(), start(), join(), close()` |
| MPI launch backend | `ProcessGroup(..., pmi=PMIBackend.CRAY)` |
| Capture rank output | `stdout=MSG_PIPE`, `Process(None, ident=puid).stdout_conn` |
| Suppress extra output | `stdout=MSG_DEVNULL` |
| File output | `stdout="file.txt"` |
| Placement control | `Policy(...)` on group or template |

### Next steps

- Replace `mpi_hello` with your production MPI executable.
- Add parser processes to extract metrics from rank-0 stdout.
- Explore template-level and group-level policies for multi-job scheduling.